# <font color='lightgreen'>🧹 02 clean data</font>
---

### <font color='lightgreen'>Librerías</font>

In [61]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

### <font color='lightgreen'>Configuración</font>

In [62]:
# Configuración
DATA_LOADED = "../data/loaded"
DATA_CLEAN = "../data/clean"

# Crear carpeta de salida
Path(DATA_CLEAN).mkdir(parents=True, exist_ok=True)

# Lista de archivos a limpiar
archivos = [
    "compras",
    "consumo_energetico",
    "encuestas",
    "eventos_rrhh",
    "personal_nomina",
    "residuos",
    "ventas"
]

# Diccionario para guardar los dataframes limpios
datos_clean = {}

### <font color='lightgreen'>Funciones de Limpieza</font>

In [63]:
def resumen_calidad(df, nombre):
    """Genera resumen de calidad del dataframe"""
    print(f"\n📊 {nombre.upper()}")
    print("-" * 40)
    print(f"  • Filas: {len(df):,}")
    print(f"  • Columnas: {len(df.columns)}")
    
    # Nulos por columna
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos) > 0:
        print(f"\n  ⚠️  Valores nulos:")
        for col, count in nulos.items():
            pct = (count / len(df)) * 100
            print(f"     - {col}: {count:,} ({pct:.1f}%)")
    else:
        print(f"\n  ✅ Sin valores nulos")
    
    # Duplicados
    duplicados = df.duplicated().sum()
    if duplicados > 0:
        print(f"\n  ⚠️  Filas duplicadas: {duplicados:,}")
    else:
        print(f"\n  ✅ Sin filas duplicadas")
    
    return nulos, duplicados

In [64]:
#This Python function `detectar_outliers` is designed to detect outliers in a DataFrame `df` for the specified numeric columns `columnas_numericas` using the Interquartile Range (IQR) method.
def detectar_outliers(df, columnas_numericas):
    """Detecta outliers usando IQR"""
    outliers = {}
    for col in columnas_numericas:
        if col in df.columns and df[col].dtype in ['int64', 'float64']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers_count = ((df[col] < lower) | (df[col] > upper)).sum()
            if outliers_count > 0:
                outliers[col] = outliers_count
    return outliers

### <font color='lightgreen'>1. Cargar Datos Raw</font>

In [65]:
print("\n" + "=" * 50)
print("CARGANDO DATOS RAW...")
print("=" * 50)

for nombre in archivos:
    file_path = f"{DATA_LOADED}/{nombre}_raw.csv"
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        datos_clean[nombre] = df
        print(f"✅ {nombre}: {len(df)} filas")
    except Exception as e:
        print(f"❌ {nombre}: {e}")


CARGANDO DATOS RAW...
✅ compras: 864 filas
✅ consumo_energetico: 178 filas
✅ encuestas: 189 filas
✅ eventos_rrhh: 537 filas
✅ personal_nomina: 1378 filas
✅ residuos: 2508 filas
✅ ventas: 43893 filas


### <font color='lightgreen'>2. Carga de datos</font>

#### <font color='lightgreen'>> 2.1 Compras Proveedor</font>

In [66]:
print("\n" + "=" * 50)
print("LIMPIEZA: COMPRAS PROVEEDOR")
print("=" * 50)

df = datos_clean['compras'].copy()

# Resumen inicial
resumen_calidad(df, "compras - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['compras']) - len(df)}")

# Validar montos positivos
if 'monto_ars' in df.columns:
    negativos = (df['monto_ars'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Montos negativos: {negativos} (se convertirán a 0)")
        df['monto_ars'] = df['monto_ars'].clip(lower=0)

# Verificar fechas
if 'fecha_emision' in df.columns:
    fechas_invalidas = df['fecha_emision'].isna().sum()
    if fechas_invalidas > 0:
        print(f"  ⚠️  Fechas inválidas: {fechas_invalidas}")

# Resumen final
resumen_calidad(df, "compras - después")
datos_clean['compras'] = df


LIMPIEZA: COMPRAS PROVEEDOR

📊 COMPRAS - ANTES
----------------------------------------
  • Filas: 864
  • Columnas: 8

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 COMPRAS - DESPUÉS
----------------------------------------
  • Filas: 864
  • Columnas: 8

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


#### <font color='lightgreen'>> 2.2 Consumo Energético</font>

In [67]:
print("\n" + "=" * 50)
print("LIMPIEZA: CONSUMO ENERGÉTICO")
print("=" * 50)

df = datos_clean['consumo_energetico'].copy()

# Resumen inicial
resumen_calidad(df, "consumo_energetico - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['consumo_energetico']) - len(df)}")

# Validar consumos positivos
if 'kwh_consumidos' in df.columns:
    negativos = (df['kwh_consumidos'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  kWh negativos: {negativos} (se convertirán a 0)")
        df['kwh_consumidos'] = df['kwh_consumidos'].clip(lower=0)

if 'm3_gas' in df.columns:
    negativos = (df['m3_gas'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  m3 gas negativos: {negativos} (se convertirán a 0)")
        df['m3_gas'] = df['m3_gas'].clip(lower=0)

# Recalcular kwh_totales si existe
if 'kwh_totales' in df.columns:
    df['kwh_totales'] = df['kwh_consumidos'].fillna(0) + (df['m3_gas'].fillna(0) * 10.69)

# Resumen final
resumen_calidad(df, "consumo_energetico - después")
datos_clean['consumo_energetico'] = df


LIMPIEZA: CONSUMO ENERGÉTICO

📊 CONSUMO_ENERGETICO - ANTES
----------------------------------------
  • Filas: 178
  • Columnas: 13

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 CONSUMO_ENERGETICO - DESPUÉS
----------------------------------------
  • Filas: 178
  • Columnas: 13

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


#### <font color='lightgreen'>> 2.3 Encuestas Clima</font>

In [68]:
print("\n" + "=" * 50)
print("LIMPIEZA: ENCUESTAS CLIMA")
print("=" * 50)

df = datos_clean['encuestas'].copy()

# Resumen inicial
resumen_calidad(df, "encuestas - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['encuestas']) - len(df)}")

# Validar rangos de satisfacción (1-10)
columnas_satisfaccion = ['satisfaccion_gral', 'clima_equipo', 'liderazgo', 'condiciones_fisicas']
for col in columnas_satisfaccion:
    if col in df.columns:
        fuera_rango = ((df[col] < 1) | (df[col] > 10)).sum()
        if fuera_rango > 0:
            print(f"  ⚠️  {col}: {fuera_rango} valores fuera de rango (1-10)")
            df[col] = df[col].clip(lower=1, upper=10)

# Resumen final
resumen_calidad(df, "encuestas - después")
datos_clean['encuestas'] = df


LIMPIEZA: ENCUESTAS CLIMA

📊 ENCUESTAS - ANTES
----------------------------------------
  • Filas: 189
  • Columnas: 12

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 ENCUESTAS - DESPUÉS
----------------------------------------
  • Filas: 189
  • Columnas: 12

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


#### <font color='lightgreen'>> 2.4 Eventos RRHH</font>

In [69]:
print("\n" + "=" * 50)
print("LIMPIEZA: EVENTOS RRHH")
print("=" * 50)

df = datos_clean['eventos_rrhh'].copy()

# Resumen inicial
resumen_calidad(df, "eventos_rrhh - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['eventos_rrhh']) - len(df)}")

# Validar horas capacitación y días baja no negativos
if 'horas_capacitacion' in df.columns:
    negativos = (df['horas_capacitacion'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Horas capacitación negativas: {negativos} (se convertirán a 0)")
        df['horas_capacitacion'] = df['horas_capacitacion'].clip(lower=0)

if 'dias_baja' in df.columns:
    negativos = (df['dias_baja'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Días baja negativos: {negativos} (se convertirán a 0)")
        df['dias_baja'] = df['dias_baja'].clip(lower=0)

# Resumen final
resumen_calidad(df, "eventos_rrhh - después")
datos_clean['eventos_rrhh'] = df


LIMPIEZA: EVENTOS RRHH

📊 EVENTOS_RRHH - ANTES
----------------------------------------
  • Filas: 537
  • Columnas: 10

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 EVENTOS_RRHH - DESPUÉS
----------------------------------------
  • Filas: 537
  • Columnas: 10

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


#### <font color='lightgreen'>> 2.5 Personal Nómina</font>

In [70]:
print("\n" + "=" * 50)
print("LIMPIEZA: PERSONAL NÓMINA")
print("=" * 50)

df = datos_clean['personal_nomina'].copy()

# Resumen inicial
resumen_calidad(df, "personal_nomina - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['personal_nomina']) - len(df)}")

# Validar salarios positivos
if 'total_devengado' in df.columns:
    negativos = (df['total_devengado'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Salarios negativos: {negativos} (se convertirán a 0)")
        df['total_devengado'] = df['total_devengado'].clip(lower=0)

# Resumen final
resumen_calidad(df, "personal_nomina - después")
datos_clean['personal_nomina'] = df


LIMPIEZA: PERSONAL NÓMINA

📊 PERSONAL_NOMINA - ANTES
----------------------------------------
  • Filas: 1,378
  • Columnas: 12

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 PERSONAL_NOMINA - DESPUÉS
----------------------------------------
  • Filas: 1,378
  • Columnas: 12

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


#### <font color='lightgreen'>> 2.6 Residuos Reciclaje</font>

In [71]:
print("\n" + "=" * 50)
print("LIMPIEZA: RESIDUOS RECICLAJE")
print("=" * 50)

df = datos_clean['residuos'].copy()

# Resumen inicial
resumen_calidad(df, "residuos - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['residuos']) - len(df)}")

# Validar kg no negativos
if 'kg_generados' in df.columns:
    negativos = (df['kg_generados'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  kg generados negativos: {negativos} (se convertirán a 0)")
        df['kg_generados'] = df['kg_generados'].clip(lower=0)

if 'kg_reciclados' in df.columns:
    negativos = (df['kg_reciclados'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  kg reciclados negativos: {negativos} (se convertirán a 0)")
        df['kg_reciclados'] = df['kg_reciclados'].clip(lower=0)

# Recalcular tasa_reciclaje
if 'kg_reciclados' in df.columns and 'kg_generados' in df.columns:
    df['tasa_reciclaje'] = (df['kg_reciclados'] / df['kg_generados'].replace(0, np.nan)) * 100
    df['tasa_reciclaje'] = df['tasa_reciclaje'].fillna(0).clip(lower=0, upper=100)

# Resumen final
resumen_calidad(df, "residuos - después")
datos_clean['residuos'] = df


LIMPIEZA: RESIDUOS RECICLAJE

📊 RESIDUOS - ANTES
----------------------------------------
  • Filas: 2,508
  • Columnas: 11

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 RESIDUOS - DESPUÉS
----------------------------------------
  • Filas: 2,508
  • Columnas: 11

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


#### <font color='lightgreen'>> 2.7 Ventas Transacciones</font>

In [72]:
print("\n" + "=" * 50)
print("LIMPIEZA: VENTAS TRANSACCIONES")
print("=" * 50)

df = datos_clean['ventas'].copy()

# Resumen inicial
resumen_calidad(df, "ventas - antes")

# Eliminar duplicados
df = df.drop_duplicates()
print(f"\n  🔄 Duplicados eliminados: {len(datos_clean['ventas']) - len(df)}")

# Validar cantidades y precios positivos
if 'cantidad' in df.columns:
    negativos = (df['cantidad'] <= 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Cantidades no positivas: {negativos} (se eliminarán)")
        df = df[df['cantidad'] > 0]

if 'precio_unitario' in df.columns:
    negativos = (df['precio_unitario'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Precios negativos: {negativos} (se convertirán a 0)")
        df['precio_unitario'] = df['precio_unitario'].clip(lower=0)

if 'subtotal_ars' in df.columns:
    negativos = (df['subtotal_ars'] < 0).sum()
    if negativos > 0:
        print(f"  ⚠️  Subtotal negativo: {negativos} (se convertirán a 0)")
        df['subtotal_ars'] = df['subtotal_ars'].clip(lower=0)

# Resumen final
resumen_calidad(df, "ventas - después")
datos_clean['ventas'] = df


LIMPIEZA: VENTAS TRANSACCIONES

📊 VENTAS - ANTES
----------------------------------------
  • Filas: 43,893
  • Columnas: 15

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas

  🔄 Duplicados eliminados: 0

📊 VENTAS - DESPUÉS
----------------------------------------
  • Filas: 43,893
  • Columnas: 15

  ✅ Sin valores nulos

  ✅ Sin filas duplicadas


### <font color='lightgreen'>3. Detección de Outliers</font>

In [73]:
print("\n" + "=" * 50)
print("DETECCIÓN DE OUTLIERS")
print("=" * 50)

outliers_report = {}

# Compras
if 'monto_ars' in datos_clean['compras'].columns:
    outliers = detectar_outliers(datos_clean['compras'], ['monto_ars'])
    if outliers:
        outliers_report['compras'] = outliers
        print(f"\n📊 compras: {outliers}")

# Ventas
if 'subtotal_ars' in datos_clean['ventas'].columns:
    outliers = detectar_outliers(datos_clean['ventas'], ['subtotal_ars', 'cantidad'])
    if outliers:
        outliers_report['ventas'] = outliers
        print(f"\n📊 ventas: {outliers}")

# Consumo energético
if 'kwh_consumidos' in datos_clean['consumo_energetico'].columns:
    outliers = detectar_outliers(datos_clean['consumo_energetico'], ['kwh_consumidos'])
    if outliers:
        outliers_report['consumo_energetico'] = outliers
        print(f"\n📊 consumo_energetico: {outliers}")

if outliers_report:
    print("\n⚠️  Se detectaron outliers. Revisar si requieren tratamiento especial.")
else:
    print("\n✅ No se detectaron outliers significativos")


DETECCIÓN DE OUTLIERS

📊 compras: {'monto_ars': np.int64(122)}

📊 ventas: {'subtotal_ars': np.int64(6808), 'cantidad': np.int64(6504)}

⚠️  Se detectaron outliers. Revisar si requieren tratamiento especial.


#### <font color='lightgreen'>4. Guardar Archivos Limpios</font>


In [74]:
print("\n" + "=" * 50)
print("GUARDANDO ARCHIVOS LIMPIOS...")
print("=" * 50)

for nombre, df in datos_clean.items():
    output_file = f"{DATA_CLEAN}/{nombre}_clean.csv"
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"   💾 {nombre}_clean.csv ({len(df)} filas, {len(df.columns)} columnas)")


GUARDANDO ARCHIVOS LIMPIOS...
   💾 compras_clean.csv (864 filas, 8 columnas)
   💾 consumo_energetico_clean.csv (178 filas, 13 columnas)
   💾 encuestas_clean.csv (189 filas, 12 columnas)
   💾 eventos_rrhh_clean.csv (537 filas, 10 columnas)
   💾 personal_nomina_clean.csv (1378 filas, 12 columnas)
   💾 residuos_clean.csv (2508 filas, 11 columnas)
   💾 ventas_clean.csv (43893 filas, 15 columnas)


### <font color='lightgreen'>5. Resumen Final</font>

In [75]:
print("\n" + "=" * 50)
print("✅ LIMPIEZA COMPLETADA")
print("=" * 50)

print(f"\n📊 Resumen de datasets limpios:")
print("-" * 50)
print(f"{'Dataset':<20} {'Filas':>10} {'Columnas':>10} {'Nulos':>10}")
print("-" * 50)

for nombre, df in datos_clean.items():
    nulos_total = df.isnull().sum().sum()
    print(f"{nombre:<20} {len(df):>10,} {len(df.columns):>10} {nulos_total:>10}")

print(f"\n📁 Archivos guardados en: {DATA_CLEAN}/")

print("\n🔍 Próximo paso: Ejecutar 03_build_star_schema.ipynb")


✅ LIMPIEZA COMPLETADA

📊 Resumen de datasets limpios:
--------------------------------------------------
Dataset                   Filas   Columnas      Nulos
--------------------------------------------------
compras                     864          8          0
consumo_energetico          178         13          0
encuestas                   189         12          0
eventos_rrhh                537         10          0
personal_nomina           1,378         12          0
residuos                  2,508         11          0
ventas                   43,893         15          0

📁 Archivos guardados en: ../data/clean/

🔍 Próximo paso: Ejecutar 03_build_star_schema.ipynb
